In [44]:
import numpy as np
import pandas as pd
from trading_rookie.data.get_data import DataDownloader

dd = DataDownloader()
spy_px = dd.yahoo("SPY", "2016-01-01", "2025-12-31")[["Open","Adj_close"]]
spy_close = spy_px["Adj_close"]
spy_open = spy_px["Open"]

close_df = pd.read_csv("outputs/close_df.csv", index_col=0)
open_df = pd.read_csv("outputs/open_df.csv", index_col=0)
close_df.index = pd.to_datetime(close_df.index)
open_df.index = pd.to_datetime(open_df.index)
universe50 = pd.read_csv("outputs/universe50.csv")

# make close_df only have 50 stocks in universe50
close_df = close_df[universe50["ticker"]]

Successfully got stock data for SPY


In [45]:
def apply_position_caps_long(w: pd.Series, pos_cap: float) -> pd.Series:
    w = w.copy()
    non_spy = w.index[w.index != "SPY"]
    w.loc[non_spy] = w.loc[non_spy].clip(0.0, pos_cap)
    if "SPY" in w.index:
        w.loc["SPY"] = 0.0
    return w


def apply_gross_cap(w: pd.Series, gross_cap: float) -> pd.Series:
    w = w.copy()
    gross = float(w[w > 0].sum())
    if gross > gross_cap and gross > 0:
        pos = w > 0
        w.loc[pos] = w.loc[pos] * (gross_cap / gross)
    return w


def estimate_portfolio_vol_from_returns(
    w: pd.Series, returns_panel: pd.DataFrame, t: pd.Timestamp, cov_lookback: int = 80
) -> float:
    cols = [c for c in w.index if w.get(c, 0.0) > 1e-12 and c in returns_panel.columns]
    if len(cols) < 2:
        return np.nan
    r = returns_panel[cols].loc[:t].tail(cov_lookback).dropna()
    if len(r) < 40:
        return np.nan
    wc = w[cols].to_numpy(dtype=float)
    cov = r.cov().to_numpy()
    var = float(wc @ cov @ wc)
    return float(np.sqrt(max(var, 0.0)) * np.sqrt(252))


def compute_defensive_features(
    close_df: pd.DataFrame, spy_close: pd.Series, spy_trend_days: int = 200
):
    """Features for long-only, trend-gated risk parity sleeve."""
    close = close_df.sort_index()
    spy_close = spy_close.sort_index()
    close, spy_close = close.align(spy_close, join="inner", axis=0)

    log_ret = np.log(close).diff()
    spy_ret = np.log(spy_close).diff()
    spy_ma = spy_close.rolling(spy_trend_days).mean()
    risk_on = spy_close > spy_ma

    # ~6m return skipping the most recent month (reduces short-term reversal)
    mom_signal = np.log(close.shift(21) / close.shift(147))
    vol20 = log_ret.rolling(20).std() * np.sqrt(252)
    sma50 = close.rolling(50).mean()
    spy_vol20 = spy_ret.rolling(20).std() * np.sqrt(252)

    returns_panel = log_ret.copy()
    returns_panel["SPY"] = spy_ret

    return {
        "close": close,
        "mom_signal": mom_signal,
        "vol20": vol20,
        "sma50": sma50,
        "risk_on": risk_on,
        "spy_vol20": spy_vol20,
        "returns_panel": returns_panel,
    }


def build_defensive_weights(
    t: pd.Timestamp,
    feats: dict,
    universe_cols: list,
    *,
    top_n: int = 14,
    vol_target: float = 0.075,
    gross_cap: float = 1.0,
    pos_cap: float = 0.10,
    cov_lookback: int = 80,
    n_iter: int = 6,
    spy_vol_soft: float = 0.18,
    spy_vol_hard: float = 0.28,
    soft_scale: float = 0.62,
    hard_scale: float = 0.35,
    dd_scale: float = 1.0,
    require_stock_trend: bool = True,
):
    """
    Flat (cash) when SPY is below its long MA. Otherwise top-N by 6m momentum,
    risk-parity-ish (1/vol), vol-targeted. SPY is not held; gate replaces beta hedge.
    """
    ix = list(universe_cols) + ["SPY"]
    flat = pd.Series(0.0, index=ix, dtype=float)

    ro = feats["risk_on"].loc[t]
    if not (pd.notna(ro) and bool(ro)):
        return flat

    mom = feats["mom_signal"].loc[t].reindex(universe_cols).dropna()
    if len(mom) < max(8, top_n // 2):
        return flat

    top = mom.sort_values(ascending=False).head(min(top_n, len(mom))).index.tolist()
    if require_stock_trend:
        px = feats["close"].loc[t, top]
        sm = feats["sma50"].loc[t, top]
        ok = (px > sm) & px.notna() & sm.notna()
        top = ok[ok].index.tolist()
        if len(top) < 4:
            return flat

    v = feats["vol20"].loc[t, top].replace([np.inf, -np.inf], np.nan).dropna()
    if len(v) < 4:
        return flat
    top = v.index.tolist()
    inv = 1.0 / v.loc[top]
    w_long = inv / inv.sum()

    w = flat.copy()
    w.loc[top] = w_long.values

    sv = feats["spy_vol20"].loc[t]
    if pd.notna(sv):
        if sv > spy_vol_hard:
            w = w * hard_scale
        elif sv > spy_vol_soft:
            w = w * soft_scale

    w = w * dd_scale

    rp = feats["returns_panel"]
    for _ in range(n_iter):
        w = apply_position_caps_long(w, pos_cap=pos_cap)
        w = apply_gross_cap(w, gross_cap=gross_cap)
        pv = estimate_portfolio_vol_from_returns(w, rp, t, cov_lookback=cov_lookback)
        if not np.isfinite(pv) or pv <= 0:
            break
        w = w * (vol_target / pv)

    w = apply_position_caps_long(w, pos_cap=pos_cap)
    w = apply_gross_cap(w, gross_cap=gross_cap)

    if float(w.sum()) < 1e-8:
        return flat
    return w.sort_index()


In [ ]:
def weights_to_target_shares(weights: pd.Series, equity: float, next_open: pd.Series) -> pd.Series:
    dollars = weights * equity
    shares = dollars / next_open
    shares = shares.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return shares

def compute_trades(target_shares: pd.Series, current_shares: pd.Series) -> pd.Series:
    idx = target_shares.index.union(current_shares.index)
    target = target_shares.reindex(idx).fillna(0.0)
    current = current_shares.reindex(idx).fillna(0.0)
    trades = target - current
    return trades

def apply_trades(current_shares: pd.Series, cash: float, trades: pd.Series, fill_prices: pd.Series,
                 fee_bps=1.0, slippage_bps=2.0):
    notional = trades * fill_prices
    cost = np.abs(notional).sum() * (fee_bps + slippage_bps) / 1e4

    new_cash = cash - notional.sum() - cost
    new_shares = current_shares.add(trades, fill_value=0.0)

    return new_shares, float(new_cash)

def mark_to_market(shares: pd.Series, cash: float, close_prices: pd.Series) -> float:
    px = close_prices.reindex(shares.index).fillna(0.0)
    equity = float((shares * px).sum() + cash)
    return equity

def run_backtest_from_panels(
    close_df: pd.DataFrame,
    open_df: pd.DataFrame,
    spy_close: pd.Series,
    spy_open: pd.Series,
    initial_cash: float = 1_000_000.0,
    *,
    spy_trend_days: int = 200,
    top_n: int = 14,
    vol_target: float = 0.2,
    cov_lookback: int = 80,
    require_stock_trend: bool = True,
    fee_bps: float = 1.0,
    slippage_bps: float = 2.0,
):
    feats = compute_defensive_features(close_df, spy_close, spy_trend_days=spy_trend_days)
    returns_panel = feats["returns_panel"]
    spy_vol20 = feats["spy_vol20"]

    dates = close_df.index.sort_values()
    universe = list(close_df.columns)

    shares = pd.Series(dtype=float)
    cash = initial_cash
    peak_equity = float(initial_cash)
    equity_records = []
    weight_records = []
    debug_records = []
    all_weight_cols = universe + ["SPY"]

    for i in range(len(dates) - 1):
        t = dates[i]
        t1 = dates[i + 1]

        eq_t = mark_to_market(shares, cash, close_df.loc[t].reindex(shares.index).fillna(0.0))
        if not np.isfinite(eq_t):
            eq_t = cash
        peak_equity = max(peak_equity, float(eq_t))
        dd = float(eq_t / peak_equity - 1.0) if peak_equity > 0 else 0.0
        if dd <= -0.14:
            dd_scale = 0.30
        elif dd <= -0.10:
            dd_scale = 0.50
        elif dd <= -0.06:
            dd_scale = 0.72
        elif dd <= -0.03:
            dd_scale = 0.88
        else:
            dd_scale = 1.0

        w = build_defensive_weights(
            t,
            feats,
            universe,
            top_n=top_n,
            vol_target=vol_target,
            gross_cap=1.0,
            pos_cap=0.20,
            cov_lookback=cov_lookback,
            n_iter=6,
            dd_scale=dd_scale,
            require_stock_trend=require_stock_trend,
        )

        # 2. prepare next open / next close
        next_open = open_df.loc[t1].copy()
        next_close = close_df.loc[t1].copy()

        next_open["SPY"] = float(spy_open.loc[t1])
        next_close["SPY"] = float(spy_close.loc[t1])

        current_equity_proxy = (
            mark_to_market(shares, cash, next_close.reindex(shares.index).fillna(0.0))
            if len(shares) > 0 else cash
        )

        target_shares = weights_to_target_shares(w, current_equity_proxy, next_open[w.index])
        trades = compute_trades(target_shares, shares)

        fill_prices = next_open.reindex(trades.index).fillna(0.0)
        shares, cash = apply_trades(
            shares, cash, trades, fill_prices, fee_bps=fee_bps, slippage_bps=slippage_bps
        )

        equity = mark_to_market(shares, cash, next_close)
        equity_records.append((t1, equity))

        w_out = w.reindex(all_weight_cols).fillna(0.0)
        w_out.name = t
        weight_records.append(w_out)

        debug_records.append({
            "Date": t,
            "gross": float(w[w > 0].sum()),
            "net": float(w.sum()),
            "spy_weight": float(w.get("SPY", 0.0)),
            "risk_on": float(bool(feats["risk_on"].loc[t]) if pd.notna(feats["risk_on"].loc[t]) else False),
            "dd_scale": dd_scale,
            "spy_vol20": float(spy_vol20.loc[t]) if pd.notna(spy_vol20.loc[t]) else np.nan,
            "est_port_vol": estimate_portfolio_vol_from_returns(w, returns_panel, t, cov_lookback=cov_lookback),
            "n_positions": int((w.reindex(close_df.columns).fillna(0.0) > 1e-8).sum()),
        })

    equity_curve = pd.DataFrame(equity_records, columns=["Date", "Equity"]).set_index("Date")
    equity_curve["ret"] = equity_curve["Equity"].pct_change().fillna(0.0)

    weights_df = pd.DataFrame(weight_records)
    debug_df = pd.DataFrame(debug_records).set_index("Date") if debug_records else pd.DataFrame()

    return equity_curve, weights_df, debug_df

In [46]:
# Goal-oriented sleeve (not guaranteed Sharpe>1 on every window): trend gate + vol targeting + DD throttle.
eq, wts, debug = run_backtest_from_panels(close_df, open_df, spy_close, spy_open)

# print(eq.tail())
# print(debug.tail())
# print(wts["SPY"].tail(10))


def compute_basic_metrics(equity_curve: pd.DataFrame):
    ret = equity_curve["ret"]
    sharpe = ret.mean() / ret.std() * np.sqrt(252) if ret.std() > 0 else np.nan
    cummax = equity_curve["Equity"].cummax()
    dd = equity_curve["Equity"] / cummax - 1.0
    ann_ret = (1 + ret.mean()) ** 252 - 1
    ann_vol = ret.std() * np.sqrt(252)
    return {
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_dd": dd.min(),
    }


def capm_alpha_beta_ann(port_ret: pd.Series, spy_close: pd.Series):
    spy_r = spy_close.reindex(port_ret.index).pct_change()
    df = pd.DataFrame({"p": port_ret, "s": spy_r}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(df) < 50:
        return np.nan, np.nan
    y = df["p"].to_numpy()
    x = df["s"].to_numpy()
    vx = np.var(x, ddof=1)
    if vx <= 0:
        return np.nan, np.nan
    beta = np.cov(x, y, ddof=1)[0, 1] / vx
    alpha_d = y.mean() - beta * x.mean()
    ann_alpha = (1 + alpha_d) ** 252 - 1
    return ann_alpha, beta


print(compute_basic_metrics(eq))
aa, b = capm_alpha_beta_ann(eq["ret"], spy_close)
print({"capm_ann_alpha": aa, "capm_beta": b})

{'ann_return': 0.11064550580077004, 'ann_vol': 0.18900836710848068, 'sharpe': 0.5553364556130803, 'max_dd': -0.15439273406247778}
{'capm_ann_alpha': 0.06646742193498456, 'capm_beta': 0.260430395052867}


In [39]:
# Optional sweeps (defensive sleeve: SPY MA gate + skip-month mom + vol targeting).


def compute_basic_metrics(equity_curve: pd.DataFrame):
    ret = equity_curve["ret"]
    sharpe = ret.mean() / ret.std() * np.sqrt(252) if ret.std() > 0 else np.nan
    cummax = equity_curve["Equity"].cummax()
    dd = equity_curve["Equity"] / cummax - 1.0
    ann_ret = (1 + ret.mean()) ** 252 - 1
    ann_vol = ret.std() * np.sqrt(252)
    return {
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_dd": dd.min(),
    }


def avg_gross_turnover(weights_df: pd.DataFrame) -> float:
    w = weights_df.fillna(0.0)
    return float(w.diff().abs().sum(axis=1).mean())


rows = []
for name, kw in [
    ("defensive default", {}),
    ("faster MA 120d", {"spy_trend_days": 120}),
    ("tighter vol 6.5%", {"vol_target": 0.065}),
    ("no stock SMA filter", {"require_stock_trend": False}),
    ("top 12 names", {"top_n": 12}),
]:
    eq_i, w_i, _ = run_backtest_from_panels(close_df, open_df, spy_close, spy_open, **kw)
    m = compute_basic_metrics(eq_i)
    m["name"] = name
    m["avg_gross_turnover"] = avg_gross_turnover(w_i)
    rows.append(m)

cmp = pd.DataFrame(rows).set_index("name")
print(cmp.round(4))

                     ann_return  ann_vol  sharpe  max_dd  avg_gross_turnover
name                                                                        
defensive default        0.0652   0.1208  0.5231 -0.1149              0.1150
faster MA 120d           0.0627   0.1314  0.4626 -0.1069              0.1169
tighter vol 6.5%         0.0552   0.1059  0.5073 -0.1034              0.1004
no stock SMA filter      0.0799   0.1063  0.7233 -0.1100              0.0802
top 12 names             0.0703   0.1297  0.5240 -0.1279              0.1099
